In [0]:
import requests, json, time
from pyspark.sql import functions as F

DRUGS = ["metformin", "atorvastatin", "lisinopril", "warfarin",
         "ibuprofen", "amlodipine", "omeprazole", "sertraline"]

def fetch(url, retries=3):
    for i in range(retries):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 404:
                return []
            r.raise_for_status()
            return r.json().get("results", [])
        except requests.RequestException as e:
            if i == retries - 1:
                print(f"FAILED: {url} — {e}")
                return []
            time.sleep(2 ** i)

def fetch_labels(drug, limit=100):
    return fetch(f"https://api.fda.gov/drug/label.json?"
                 f"search=openfda.generic_name:{drug}&limit={limit}")

def fetch_events(drug, limit=100):
    return fetch(f"https://api.fda.gov/drug/event.json?"
                 f"search=patient.drug.medicinalproduct:{drug}&limit={limit}")

records = []
for d in DRUGS:
    print(f"Fetching {d}...")
    for label in fetch_labels(d):
        records.append({"drug": d, "source": "label", "payload": json.dumps(label)})
    time.sleep(1)
    for ev in fetch_events(d):
        records.append({"drug": d, "source": "event", "payload": json.dumps(ev)})
    time.sleep(1)

print(f"Total records: {len(records)}")

df = (spark.createDataFrame(records)
        .withColumn("ingested_at", F.current_timestamp()))

# Bronze is append-only; Silver deduplicates by safetyreportid / drug_generic+manufacturer
(df.write.mode("append")
   .saveAsTable("fda_rag.bronze.openfda_raw"))

print("Bronze counts by source:")
spark.sql("SELECT source, COUNT(*) FROM fda_rag.bronze.openfda_raw GROUP BY source").show()